# Non-uniform BC sensitivity test

Every forcing-sensitivity check so far (`Forcing sensitivity (T8)` in the sibling notebooks) scales
**all 7** source locations by the same factor at once. That can't tell us whether the model actually
attends to *which* node each inflow lands on, or whether it just responds to some aggregate/broadcast
signal.

This notebook probes that with 7 scenarios that each perturb a subset of the locations while holding
the rest fixed (`Kirmutscheid` is the main stem, mean 70.0 m³/s — about 4x the next largest;
`new1`/`Niederadenau`/`new2` are the smallest tributaries, mean 10.4/10.5/5.0 m³/s):

- `baseline_1x` — reference, all locations at 1x
- `tributaries_0x` — smallest tributaries zeroed, rest 1x (expect small change: minor contributors)
- `tributaries_2x` — smallest tributaries doubled, rest 1x (opposite direction of the above)
- `mainstem_0x` — main stem zeroed, rest 1x (opposite target: attack the dominant source instead)
- `mainstem_2x` — main stem doubled, rest 1x (localized surge test)
- `only_mainstem` — only the main stem active, everything else 0x (isolation test)
- `only_smallest_tributary` — only the smallest tributary active, everything else 0x (reverse isolation test)

**Expectation:** the tributary scenarios should move the flood only a little; the main-stem scenarios
should move it a lot. A flat response to `mainstem_0x`/`mainstem_2x`, or a large response to the
tributary scenarios, would mean the model isn't actually using per-location BC information correctly.

Uses the best-converged checkpoint from `best_sweep_multisim_multiscale_loss`
(`best_valloss`, 4-scale mesh), same test event as
`visualize_finetuned_fixed_gt_multisim_multiscale_loss.ipynb`.

In [1]:
import sys, os

_proj_db = r'C:\Users\marrocol\AppData\Local\miniforge3\envs\mswe-gnn\Lib\site-packages\pyproj\proj_dir\share\proj'
os.environ.setdefault('PROJ_DATA', _proj_db)
os.environ.setdefault('PROJ_LIB',  _proj_db)

# Resolve the repo root robustly (works in VS Code and nbconvert, any start cwd)
try:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '..'))
except NameError:
    _here = os.getcwd()
    REPO_ROOT = _here if os.path.isdir(os.path.join(_here, 'database')) \
                else os.path.abspath(os.path.join(_here, '..'))
assert os.path.isdir(os.path.join(REPO_ROOT, 'database')), f'Not the repo root: {REPO_ROOT}'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
import wandb
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from utils.load import read_config
from utils.miscellaneous import get_model, fix_dict_in_config, get_CSI
from utils.dataset import create_model_dataset, get_temporal_test_dataset_parameters, to_temporal_dataset, separate_multiscale_node_features
from utils.visualization import PlotRollout
from training.train import LightningTrainer, rollout_test_warmstart

torch.backends.cudnn.deterministic = True
torch.set_float32_matmul_precision('high')
print('Repo root:', REPO_ROOT)

Repo root: c:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\mSWE-GNN_marg


## Load config, dataset, and checkpoint

In [2]:
CONFIG       = 'config_best_sweep.yaml'   # architecture auto-detected per checkpoint; 4-scale mesh
DATASET_NAME = 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart'   # held-out 1.0x event, 4-scale mesh
CHECKPOINT   = os.path.join(REPO_ROOT, 'results', 'best_sweep_multisim_multiscale_loss.h5')  # best_valloss (well-converged; see multiscale_loss notebook diagnostic)

exists = os.path.exists(CHECKPOINT)
size = f"{os.path.getsize(CHECKPOINT)/1e6:.1f} MB" if exists else 'MISSING'
print(f"checkpoint exists={exists!s:<6} {size:>10}  {CHECKPOINT}")

checkpoint exists=True       2.5 MB  c:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\mSWE-GNN_marg\results\best_sweep_multisim_multiscale_loss.h5


In [3]:
cfg = read_config(CONFIG)
cfg['dataset_parameters']['test_dataset_name']  = DATASET_NAME
cfg['dataset_parameters']['train_dataset_name'] = DATASET_NAME

wandb.init(mode='disabled', project='mswe-gnn', config=cfg)
fix_dict_in_config(wandb)
config = wandb.config

device = torch.device('cpu')

_, _, test_dataset, scalers = create_model_dataset(
    scalers=config.scalers, device=device,
    **config.dataset_parameters,
    **config.selected_node_features,
    **config.selected_edge_features
)

temporal_test_dataset_parameters = get_temporal_test_dataset_parameters(
    config, config.temporal_dataset_parameters
)

temporal_test_dataset = to_temporal_dataset(
    test_dataset, rollout_steps=-1, **temporal_test_dataset_parameters
)

num_node_features = temporal_test_dataset[0].x.size(-1)
num_edge_features = temporal_test_dataset[0].edge_attr.size(-1)

print('Test size:', len(test_dataset))
print('WD shape: ', test_dataset[0].WD.shape)
print('BC shape: ', temporal_test_dataset[0].BC.shape)  # [n_bc_locations, time]

The validation dataset you are using is the training one. Careful!
Test size: 1
WD shape:  torch.Size([30079, 119])
BC shape:  torch.Size([7, 3, 119])


In [4]:
def load_model_from_checkpoint(checkpoint_path):
    model_parameters = dict(config.models)
    model_type = model_parameters.pop('model_type')
    if model_type == 'MSGNN':
        model_parameters['num_scales'] = test_dataset[0].mesh.num_meshes

    ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    sd   = ckpt['state_dict']
    hid  = sd['model.edge_encoder.0.weight'].shape[0]

    proc_ids = sorted(set(
        int(k.split('.')[2]) for k in sd
        if k.startswith('model.gnn_processor.') and 'filter_matrix.' in k
    ))
    K_list = [
        sum(1 for k in sd if f'model.gnn_processor.{i}.filter_matrix.' in k and k.endswith('.weight')) - 1
        for i in proc_ids
    ]
    model_parameters['hid_features'] = hid
    model_parameters['K']            = K_list

    model = get_model(model_type)(
        num_node_features=num_node_features,
        num_edge_features=num_edge_features,
        previous_t=temporal_test_dataset_parameters['previous_t'],
        device=device,
        **model_parameters
    ).to(device)

    plmodule_kwargs = {
        'model': model,
        'lr_info': config['lr_info'],
        'trainer_options': config.trainer_options,
        'temporal_test_dataset_parameters': temporal_test_dataset_parameters
    }
    plmodule = LightningTrainer.load_from_checkpoint(
        checkpoint_path, map_location=device, **plmodule_kwargs
    )
    model = plmodule.model.to(device)
    model.eval()
    return model, hid, K_list


model, hid, K_list = load_model_from_checkpoint(CHECKPOINT)
print(f'Loaded model: hid={hid} K={K_list}')

Loaded model: hid=32 K=[1, 1, 1, 5, 4, 3, 2]


## BC location order and tributary selection

`temporal_test_dataset[0].BC` rows follow the same order as `sfincs.src` (verified in
`database/convert_sfincs_to_pkl_marg.py`: `parse_src_file` preserves file order, and the discharge
columns are asserted to match `len(src_xy)` 1:1). Read the raw `.src`/`.dis` files here to recover
the names and per-location discharge stats, and confirm the row count matches `BC.shape[0]`.

In [ ]:
SRC_FILE = 'database/raw_datasets_ahr/Simulations/ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/sfincs.src'
DIS_FILE = 'database/raw_datasets_ahr/Simulations/ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart/sfincs.dis'

names = []
with open(SRC_FILE, encoding='latin-1') as f:
    for line in f:
        if line.strip():
            names.append(line.split('"')[1])

dis = np.loadtxt(DIS_FILE)
assert dis.shape[1] - 1 == len(names) == temporal_test_dataset[0].BC.shape[0], \
    'BC row count does not match sfincs.src — order assumption is unsafe, stop and investigate'

# Some source columns contain real NaNs that the proper parser (parse_dis_file in
# database/convert_sfincs_to_pkl_marg.py) interpolates over before they reach the model.
# Total volume (integral of Q over time) is the physically relevant quantity for identifying
# the main stem vs. tributaries — not the instantaneous mean, which is more sensitive to how the
# NaN gaps happen to be distributed in time. Interpolate over NaNs before integrating.
time_s = dis[:, 0]
volumes = []
for i in range(len(names)):
    col = dis[:, i + 1]
    valid = ~np.isnan(col)
    col_filled = np.interp(time_s, time_s[valid], col[valid])
    volumes.append(np.trapezoid(col_filled, time_s))  # m^3
volumes = np.array(volumes)

print(f"{'#':>2} {'name':<14} {'mean [m3/s]':>12} {'max [m3/s]':>11} {'#NaN':>6} {'volume [Mm3]':>13}")
for i, n in enumerate(names):
    col = dis[:, i + 1]
    print(f"{i:>2} {n:<14} {np.nanmean(col):>12.2f} {np.nanmax(col):>11.2f} {np.isnan(col).sum():>6} {volumes[i]/1e6:>13.2f}")

order = np.argsort(volumes)  # ascending: smallest total contributor first
MAIN_IDX = int(order[-1])
ZERO_IDX = [int(i) for i in order[:3]]     # 3 smallest by total volume
SMALLEST_IDX = int(order[0])

print(f"\nMain stem (largest volume): {names[MAIN_IDX]}")
print(f"Smallest tributaries (3 lowest volume): {[names[i] for i in ZERO_IDX]}")
print(f"Single smallest tributary: {names[SMALLEST_IDX]}")

## Run all BC scenarios

Ground truth (`gt_s0`) is the real 1.0x SFINCS event and is unaffected by these BC edits — only the
model's input forcing changes. Comparing every scenario's prediction to the same truth isolates the
effect of each perturbation.

In [ ]:
td = temporal_test_dataset[0]
node_ptr = test_dataset[0].node_ptr
gt_s0 = separate_multiscale_node_features(td.y.detach(), node_ptr)[0]
ws_gt = gt_s0[:, 0, :].clamp(min=0).sum(0)
t_pk = int(ws_gt.argmax())

n_bc = td.BC.shape[0]

def mult(base, overrides=()):
    """overrides: list of (index_list, value) pairs applied on top of a uniform `base`.
    Shape (n_bc, 1, 1) to broadcast against td.BC's [n_bc, previous_t, time] layout."""
    m = torch.full((n_bc, 1, 1), float(base))
    for idx, val in overrides:
        m[idx] = val
    return m

scenarios = {
    'baseline_1x':             mult(1.0),
    'tributaries_0x':          mult(1.0, [(ZERO_IDX,      0.0)]),
    'tributaries_2x':          mult(1.0, [(ZERO_IDX,      2.0)]),
    'mainstem_0x':             mult(1.0, [([MAIN_IDX],    0.0)]),
    'mainstem_2x':             mult(1.0, [([MAIN_IDX],    2.0)]),
    'only_mainstem':           mult(0.0, [([MAIN_IDX],    1.0)]),
    'only_smallest_tributary': mult(0.0, [([SMALLEST_IDX], 1.0)]),
}

rows = {}
preds = {}
for name, m in scenarios.items():
    tdf = td.clone()
    tdf.BC = td.BC * m
    with torch.no_grad():
        pred = rollout_test_warmstart(model, tdf, warmup_steps=0).detach()
    p_s0 = separate_multiscale_node_features(pred, node_ptr)[0]
    preds[name] = p_s0
    csi005 = get_CSI(p_s0, gt_s0, water_threshold=0.05).nanmean().item()
    csi03  = get_CSI(p_s0, gt_s0, water_threshold=0.30).nanmean().item()
    ws = p_s0[:, 0, :].clamp(min=0).sum(0)
    rows[name] = dict(csi005=csi005, csi03=csi03, ws_peak=float(ws[t_pk]), ws_end=float(ws[-1]),
                       maxWD=float(p_s0[:, 0, :].max()))

print(f"ground truth (1x): WDsum@peak={ws_gt[t_pk]:.0f}   WDsum@end={ws_gt[-1]:.0f}   maxWD={gt_s0[:,0,:].max():.2f} m\n")

b = rows['baseline_1x']
header = f"{'scenario':<26} {'CSI@0.05':>9} {'CSI@0.30':>9} {'WDsum@peak':>11} {'d_peak%':>8} {'WDsum@end':>10} {'d_end%':>7}"
print(header)
print('-' * len(header))
for name, r in rows.items():
    d_peak = (r['ws_peak'] / b['ws_peak'] - 1) * 100 if name != 'baseline_1x' else 0.0
    d_end  = (r['ws_end']  / b['ws_end']  - 1) * 100 if name != 'baseline_1x' else 0.0
    print(f"{name:<26} {r['csi005']:>9.4f} {r['csi03']:>9.4f} {r['ws_peak']:>11.0f} {d_peak:>+7.2f}% {r['ws_end']:>10.0f} {d_end:>+6.2f}%")

### Spatial difference at the flood peak
One figure per scenario (excluding baseline): the scenario's own flood map, plus its difference
against `baseline_1x`. Tributary scenarios should show faint, localized differences; main-stem and
isolation scenarios should show large, catchment-wide differences.

In [ ]:
n_s0 = gt_s0.shape[0]
pos_s0 = np.asarray(test_dataset[0].mesh.face_xy)[:n_s0]
wd_base = preds['baseline_1x'][:, 0, t_pk].cpu().numpy()

for name, p_s0 in preds.items():
    if name == 'baseline_1x':
        continue
    wd = p_s0[:, 0, t_pk].cpu().numpy()
    diff = wd - wd_base

    fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
    masked = np.where(wd > 0.05, wd, np.nan)
    axes[0].scatter(pos_s0[:, 0], pos_s0[:, 1], c='0.92', s=4, marker='s', linewidths=0)
    sc0 = axes[0].scatter(pos_s0[:, 0], pos_s0[:, 1], c=masked, cmap='Blues', vmin=0, vmax=1.0, s=4, marker='s', linewidths=0)
    axes[0].set_title(f'{name} @ peak (t={t_pk})  wet cells: {(wd > 0.05).sum()}')
    axes[0].set_aspect('equal'); axes[0].set_xticks([]); axes[0].set_yticks([])
    plt.colorbar(sc0, ax=axes[0], shrink=0.7, label='WD [m]')

    vmax = max(abs(diff.min()), abs(diff.max()), 1e-3)
    sc1 = axes[1].scatter(pos_s0[:, 0], pos_s0[:, 1], c=diff, cmap='RdBu_r', vmin=-vmax, vmax=vmax, s=4, marker='s', linewidths=0)
    axes[1].set_title(f'diff ({name} - baseline_1x)')
    axes[1].set_aspect('equal'); axes[1].set_xticks([]); axes[1].set_yticks([])
    plt.colorbar(sc1, ax=axes[1], shrink=0.7, label='ΔWD [m]')
    plt.tight_layout()
    plt.show()

## Reading the results

- **`tributaries_0x` / `tributaries_2x`**: small CSI/volume deltas and a faint, localized diff map
  near the `new1`/`Niederadenau`/`new2` confluences is the expected, physically-consistent outcome.
- **`mainstem_0x` / `mainstem_2x`**: should show large, catchment-wide deltas — if these come back
  nearly flat like the tributary scenarios, the model is not actually responding to the dominant
  BC signal (a much bigger problem than fringe FP/FN issues).
- **`only_mainstem` / `only_smallest_tributary`**: check that a single active source produces a
  plausible, spatially-localized flood rather than leaking into unrelated parts of the graph — a
  basin-wide response from `only_smallest_tributary` in particular would point to BC signal leakage
  across nodes rather than a real forcing effect.

## Per-location sensitivity ladder (leave-one-out)

The scenarios above only probe the two extremes (main stem, the group of 3 smallest tributaries).
This ladder zeros and doubles **each of the 7 locations individually**, one at a time, rest at 1x —
covering `Denn`, `Muesch`, `Kreuzberg`, `Niederadenau`, `new1` on their own too, not just lumped into
a group. If the model is well-calibrated, its sensitivity ranking (how much CSI/volume moves when a
location is perturbed) should roughly track each location's actual share of total catchment volume
(computed in the cell above). A location with a tiny physical volume share but a large model
response (or vice versa) would be a concrete, named miscalibration to dig into.

Reported as a compact table + bar chart rather than full spatial maps for all 14 runs — see the
scenario section above for spatial detail on the two extremes.

In [ ]:
ladder_rows = []
for i, n in enumerate(names):
    for factor, tag in [(0.0, 'zero'), (2.0, 'double')]:
        m = mult(1.0, [([i], factor)])
        tdf = td.clone()
        tdf.BC = td.BC * m
        with torch.no_grad():
            pred = rollout_test_warmstart(model, tdf, warmup_steps=0).detach()
        p_s0 = separate_multiscale_node_features(pred, node_ptr)[0]
        csi005 = get_CSI(p_s0, gt_s0, water_threshold=0.05).nanmean().item()
        ws = p_s0[:, 0, :].clamp(min=0).sum(0)
        d_peak = (float(ws[t_pk]) / b['ws_peak'] - 1) * 100
        d_end  = (float(ws[-1])  / b['ws_end']  - 1) * 100
        ladder_rows.append(dict(name=n, tag=tag, idx=i, csi005=csi005, d_csi005=csi005 - b['csi005'],
                                 d_peak=d_peak, d_end=d_end, volume_share=volumes[i] / volumes.sum() * 100))

print(f"{'location':<14} {'perturb':<7} {'CSI@0.05':>9} {'dCSI@0.05':>10} {'d_peak%':>8} {'d_end%':>7} {'vol_share%':>10}")
print('-' * 76)
for r in ladder_rows:
    print(f"{r['name']:<14} {r['tag']:<7} {r['csi005']:>9.4f} {r['d_csi005']:>+10.4f} "
          f"{r['d_peak']:>+7.2f}% {r['d_end']:>+6.2f}% {r['volume_share']:>9.2f}%")

In [ ]:
sort_order = np.argsort(volumes)[::-1]  # descending physical volume share
plot_names = [names[i] for i in sort_order]
vol_share_sorted = volumes[sort_order] / volumes.sum() * 100

def get_d_peak(idx, tag):
    return next(r['d_peak'] for r in ladder_rows if r['idx'] == idx and r['tag'] == tag)

zero_d_peak   = [abs(get_d_peak(i, 'zero'))   for i in sort_order]
double_d_peak = [abs(get_d_peak(i, 'double')) for i in sort_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(plot_names))
for ax, vals, title in zip(axes, [zero_d_peak, double_d_peak], ['zeroed (0x)', 'doubled (2x)']):
    ax2 = ax.twinx()
    ax.bar(x, vals, color='steelblue', alpha=0.8)
    ax2.plot(x, vol_share_sorted, 'o-', color='darkorange')
    ax.set_xticks(x); ax.set_xticklabels(plot_names, rotation=30, ha='right')
    ax.set_ylabel('|ΔWDsum@peak| [%]', color='steelblue')
    ax2.set_ylabel('volume share [%]', color='darkorange')
    ax.tick_params(axis='y', labelcolor='steelblue')
    ax2.tick_params(axis='y', labelcolor='darkorange')
    ax.set_title(f'Sensitivity when each location is {title}\n(locations sorted by volume share, high to low)')
    ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Reading this: bars (model's response) should roughly track the orange line (actual physical\n"
      "contribution). A location with a tall bar but a low orange dot (or vice versa) is a named\n"
      "location where the model over- or under-weights that BC relative to its real contribution.")